# Dual-Basis Segmented-Arbitrage Carry Portfolio (DB-SACP)

This notebook is now wired to **one required input file**:
`data/tips_treasury_implied_rf_2010.parquet` and specifically its **4 `arb_*` columns**.

It provides fail-fast schema checks, unit checks, alignment diagnostics, EWMA risk, constrained sizing, and in-notebook tests.


## Data Checklist

### Required file
- `data/tips_treasury_implied_rf_2010.parquet`

### Required columns
- `date`
- exactly **four** columns that start with `arb_` (example: `arb_2y`, `arb_5y`, `arb_10y`, `arb_20y`)

### Units and interpretation
- `arb_*` values must be decimal rates/spreads (e.g., `0.01 = 1%`).
- If values appear percent-like (e.g., `1.0 = 1%`), set `AUTO_CONVERT_PERCENT_TO_DECIMAL=True` to divide by 100.

### Usage mapping
- `arb_*` columns are treated as the strategy dislocation series used directly for spread, z-score, vol, and sizing inputs.


## Quickstart

1. Place `tips_treasury_implied_rf_2010.parquet` in `data/`.
2. Run all cells top-to-bottom.
3. If unit check fails due to percent inputs, toggle `AUTO_CONVERT_PERCENT_TO_DECIMAL=True`.

> Demo mode was removed by request. This notebook runs only on the real parquet input.


In [ ]:
from __future__ import annotations
import math
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_FILE = Path('data/tips_treasury_implied_rf_2010.parquet')
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

AUTO_CONVERT_PERCENT_TO_DECIMAL = True
EWMA_LAMBDA = 0.97
EWMA_COV_LAMBDA = 0.985
RIDGE = 1e-8
Z_ENTRY = 1.25
Z_EXIT = 0.50
W_CAP = 0.35

EQUITY_USD = 10_000_000.0
LEVERAGE_CAP_GROSS = 4.0
LIQUIDITY_BUFFER_PCT = 0.30
RESOURCE_RATE_PER_WEIGHT = 0.06  # simplified placeholder usage rate


In [ ]:
def load_arb_data(path: Path, auto_convert: bool = True):
    if not path.exists():
        raise FileNotFoundError(f"Missing required input file: {path.resolve()}")

    df = pd.read_parquet(path)
    if 'date' not in df.columns:
        raise ValueError("Input must contain 'date' column")

    arb_cols = sorted([c for c in df.columns if c.startswith('arb_')])
    if len(arb_cols) != 4:
        raise ValueError(f"Expected exactly 4 arb_* columns, found {len(arb_cols)}: {arb_cols}")

    out = df[['date'] + arb_cols].copy()
    out['date'] = pd.to_datetime(out['date'], errors='coerce')
    if out['date'].isna().any():
        raise ValueError(f"Unparseable dates found: {int(out['date'].isna().sum())}")
    out = out.sort_values('date')
    if out['date'].duplicated().any():
        raise ValueError(f"Duplicate dates found: {int(out['date'].duplicated().sum())}")
    out = out.set_index('date')

    for c in arb_cols:
        out[c] = pd.to_numeric(out[c], errors='coerce')

    vals = out[arb_cols].stack().dropna()
    if vals.empty:
        raise ValueError('arb_* values are empty after numeric coercion')

    q99 = float(vals.abs().quantile(0.99))
    note = 'units validated as decimals'
    if q99 > 1.0:
        if auto_convert:
            out[arb_cols] = out[arb_cols] / 100.0
            note = f'percent-like detected (|q99|={q99:.3f}); auto-converted /100'
        else:
            raise ValueError(f'Percent-like units detected (|q99|={q99:.3f}); convert to decimals.')

    vals2 = out[arb_cols].stack().dropna()
    if float(vals2.min()) < -0.50 or float(vals2.max()) > 0.50:
        raise ValueError(f"Unit sanity failed after conversion: range [{float(vals2.min()):.4f}, {float(vals2.max()):.4f}]")

    tenor_map = {}
    for c in arb_cols:
        m = re.search(r'(\d+)y', c)
        tenor_map[c] = int(m.group(1)) if m else c

    return out, arb_cols, tenor_map, note


def ewma_vol(x: pd.Series, lam: float):
    dx = x.diff()
    v = 0.0
    out = []
    for i, val in enumerate(dx.values):
        if i == 0 or np.isnan(val):
            out.append(np.nan)
            continue
        v = lam * v + (1.0 - lam) * float(val * val)
        out.append(math.sqrt(max(v, 0.0)))
    return pd.Series(out, index=x.index)


def rolling_z(x: pd.Series, win: int = 252):
    m = x.rolling(win, min_periods=max(20, win // 3)).mean()
    s = x.rolling(win, min_periods=max(20, win // 3)).std(ddof=0).replace(0.0, np.nan)
    return (x - m) / s


def ewma_cov_latest(dX: pd.DataFrame, lam: float = 0.985, ridge: float = 1e-8):
    X = dX.dropna().values
    if len(X) < 2:
        raise ValueError('Not enough observations to estimate covariance')
    n = X.shape[1]
    cov = np.zeros((n, n))
    for row in X:
        v = row.reshape(-1, 1)
        cov = lam * cov + (1.0 - lam) * (v @ v.T)
    return cov + ridge * np.eye(n)


In [ ]:
# Run pipeline
arb_df, arb_cols, tenor_map, unit_note = load_arb_data(DATA_FILE, auto_convert=AUTO_CONVERT_PERCENT_TO_DECIMAL)

# Alignment diagnostics (single-source but still explicit)
diag = pd.DataFrame([
    {
        'rows': len(arb_df),
        'first_date': arb_df.index.min(),
        'last_date': arb_df.index.max(),
        'monotonic_index': bool(arb_df.index.is_monotonic_increasing),
        'duplicate_dates': int(arb_df.index.duplicated().sum()),
    }
], index=['tips_treasury_implied_rf_2010.parquet'])

missingness = arb_df[arb_cols].isna().mean().rename('missing_pct').to_frame()

S = arb_df[arb_cols].copy()  # strategy spread/dislocation input
mu = S.abs()  # simple carry proxy from dislocation magnitude

vol = pd.DataFrame({c: ewma_vol(S[c], EWMA_LAMBDA) for c in arb_cols}, index=S.index)
z = pd.DataFrame({c: rolling_z(S[c]) for c in arb_cols}, index=S.index)
cone = vol.rolling(252, min_periods=84).quantile(0.90)
riskoff = vol > cone

sig = pd.DataFrame(0.0, index=S.index, columns=arb_cols)
for c in arb_cols:
    enter = (mu[c] > 0) & (z[c].abs() >= Z_ENTRY) & (~riskoff[c].fillna(False))
    exit_ = (z[c].abs() <= Z_EXIT) | (riskoff[c].fillna(False))
    active = False
    out = []
    for t in S.index:
        if (not active) and bool(enter.loc[t]):
            active = True
        elif active and bool(exit_.loc[t]):
            active = False
        out.append(1.0 if active else 0.0)
    sig[c] = out

# Risk + weights
dS = S.diff()
Sigma = ewma_cov_latest(dS, lam=EWMA_COV_LAMBDA, ridge=RIDGE)
mu_t = mu.iloc[-1].values
w_raw = np.linalg.solve(Sigma, mu_t)
w = np.clip(w_raw, 0.0, None)
if w.sum() > 0:
    w = w / w.sum()
w = np.minimum(w, W_CAP)
if w.sum() > 0:
    w = w / w.sum()
weights = pd.Series(w, index=arb_cols, name='weight')

usage = float((weights.clip(lower=0) * RESOURCE_RATE_PER_WEIGHT).sum())
avail = EQUITY_USD * (1.0 - LIQUIDITY_BUFFER_PCT)
max_notional = 0.0 if usage <= 0 else avail / usage
max_notional = min(max_notional, LEVERAGE_CAP_GROSS * EQUITY_USD)

print('Validation note:', unit_note)
print('Loaded arb columns:', arb_cols)
print('Pipeline complete. N rows:', len(S))


In [ ]:
print('--- Validation Summary ---')
display(diag)
print('\n--- Missingness by arb series ---')
display(missingness)
print('\n--- Sizing diagnostics ---')
print(f'usage={usage:.4%} | avail=${avail:,.0f} | max_notional=${max_notional:,.0f}')
display(weights.sort_values(ascending=False))


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for c in arb_cols:
    axes[0].plot(S.index, 10_000 * S[c], label=c)
    axes[1].plot(z.index, z[c], label=c)
axes[0].set_title('arb_* dislocations (bp)')
axes[1].set_title('z-scores')
for ax in axes:
    ax.grid(True)
    ax.legend(ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
# Targeted in-notebook tests (assert based)

# 1) schema presence
assert 'date' not in arb_df.columns
assert len(arb_cols) == 4
assert all(c.startswith('arb_') for c in arb_cols)

# 2) alignment behavior (single source => intersection == index)
idx_union = arb_df.index
idx_intersection = arb_df.index
assert idx_intersection.equals(idx_union)

# 3) spread construction correctness (S should equal input arb columns)
t0 = S.index[min(10, len(S)-1)]
c0 = arb_cols[0]
assert np.isclose(S.loc[t0, c0], arb_df.loc[t0, c0], equal_nan=True)

# 4) EWMA vol non-negativity / NaN handling
vt = ewma_vol(pd.Series([1.0, 1.1, np.nan, 1.08]), 0.97)
assert (vt.dropna() >= 0).all()
assert vt.isna().sum() >= 1

# 5) weight constraints
assert (weights >= -1e-12).all()
assert (weights <= W_CAP + 1e-10).all()
if weights.sum() > 0:
    assert np.isclose(weights.sum(), 1.0)

print('All notebook tests passed.')


## Final Notes

### What changed
- Removed demo mode entirely.
- Rewired notebook to use `data/tips_treasury_implied_rf_2010.parquet` and exactly 4 `arb_*` columns as the strategy input universe.
- Kept fail-fast validation, unit checks, EWMA risk, constrained sizing, diagnostics, and assert-based tests.

### Remaining placeholders
- Resource usage model uses a simplified scalar rate (`RESOURCE_RATE_PER_WEIGHT`).
- Signal/risk parameters remain research defaults.

### Production realism still required
- Desk-calibrated funding/margin model and transaction cost assumptions.
- Optional richer decomposition (carry vs mark-to-market) if a PnL section is added.
